# EDS full-sequence stride comparison
Run all cells. Auto mode reuses each completed score and checks input file signatures. Plot-only reads the previous report for this selection. Stride 1 uses the original full-frame predictions. Every method and stride is scored on the same half-open interval for a sequence.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from functools import cached_property
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "experiments/score_windows.py").is_file())
sys.path.insert(0, str(ROOT / "experiments"))
from score_windows import Window, bench

METRICS = ["chamfer_full", "event_count_ratio", "event_rate", "iei_distance", "fano_factor"]
METHODS = ["v2e", "iebcs", "dvs_voltmeter", "v2v", "evis_core", "fracevent", "evtween_v5.2_student"]
SEQUENCES = ["02_rocket_earth_light", "06_ziggy_and_fuzz", "10_office", "14_ziggy_in_the_arena"]
EXPERIMENTS = {1: None, 2: "stride_m1", 3: "stride_m2", 4: "stride_m3"}
MODE = "auto"  # "auto" or "plot_only"
DATA = bench.v2.old.setting("EVTWEEN_DATA", bench.v2.old.LAB / "evtween/data")
DEVICE, WORKERS, VOXEL_BINS = "cpu", 16, 160
REPORT = ROOT / "reports/experiments/eds/test/full_sequence_strides"

## Common evaluation interval and scoring
The endpoint is the earliest final selected timestamp among all four conditions. Count-based frame bins use the original timeline within this shared interval. Native V2V signed voxel planes cannot recover event timestamps or polarity-separated counts and are marked unavailable for these event-stream protocols. ISI distribution is retained as an array; its scalar view is the per-sequence median ISI in ms.

In [ ]:
def options(metric):
    args = SimpleNamespace(dataset="eds", split="test", metric=metric, device=DEVICE, workers=WORKERS,
                           chunk_size=131072 if metric in bench.FULL_CHAMFERS else 4096, max_points=20000,
                           voxel_temporal_bins=VOXEL_BINS, legacy_frame_voxels=False, voxel_block_size=1,
                           count_weight=1., rate_bin_s=.01)
    return bench.settings(args)


def measure(metric, events, window, opts):
    edges, meta = window.geometry
    if metric in bench.FULL_CHAMFERS:
        kw = {key: opts[key] for key in ("device", "workers", "chunk_size", "v_px_per_ms")}
        if metric == "chamfer_count":
            result = bench.metrics_our.chamfer_count(events, window.gt, meta["width"], edges,
                                                     count_weight=opts["count_weight"], **kw)
        else:
            if metric == "chamfer_penalty":
                kw["height"] = meta["height"]
            result = getattr(bench.m, metric)(events, window.gt, meta["width"], edges[-1], **kw)
        numbers = {k: float(v) if np.isfinite(v) else None for k, v in result.items()}
        status = "ok" if all(numbers[k] is not None for k in ("chamfer_on", "chamfer_off")) else "partial_polarity"
        return numbers, {}, status if numbers["value"] is not None else "undefined"
    if metric in ("voxel_mae", "voxel_rmse"):
        bins = np.linspace(0, edges[-1], opts["temporal_bins"] + 1)
        def planes(stream):
            return bench.voxel_planes(stream, bins, meta["height"], meta["width"], by_polarity=True)
        result = getattr(bench.voxel_metric, metric + "_by_polarity")(planes(events), planes(window.gt))
        return dict(result), {}, "ok"
    value, arrays, status = bench.score(metric, events, window, opts)
    if metric == "isi_distribution" and arrays["isi_ms"].size:
        value = np.median(arrays["isi_ms"])
    return {"value": float(value) if value is not None and np.isfinite(value) else None}, arrays, status


def input_dir(method, name, step, indices):
    if step == 1:
        return ROOT / "data_test_evt/eds/test" / method / name
    return (ROOT / "data_test_exp/eds" / method / name / EXPERIMENTS[step] /
            f"k_{len(indices)}" / f"m_{step-1}" / f"frames_0_{indices[-1]}")


def run_scores():
    configs = {metric: options(metric) for metric in METRICS}
    code_id = [1, hashlib.sha256((ROOT / "experiments/score_windows.py").read_bytes()).hexdigest()]
    records = []
    for name in SEQUENCES:
        source = DATA / "eds/test" / name
        files, ts, _, base, _ = bench.v2.sequence_info(source)
        sets = {step: np.arange(0, len(ts), step) for step in EXPERIMENTS}
        last = min(ids[-1] for ids in sets.values())
        if last < 1:
            raise ValueError(f"No common frame interval for {name}")
        window = Window((source, files, ts, base), np.arange(last+1), {})
        stop = window.geometry[0][-1]
        gt_stamps = [bench.stamp(source / f) for f in ("frame_ts.npy", "ev_t.npy", "ev_x.npy", "ev_y.npy", "ev_p.npy")]
        reused = computed = 0
        for step, ids in sets.items():
            for method in [*METHODS, "ground_truth"]:
                folder = input_dir(method, name, step, ids)
                path = folder / ("voxels.npz" if method == "v2v" else "events.npz")
                metadata = folder / "input.json"
                ground = method == "ground_truth"
                stamps = gt_stamps if ground else [*gt_stamps, bench.stamp(metadata), bench.stamp(path)]
                events = None
                for metric in METRICS:
                    if ground and metric not in bench.DESCRIPTIVE:
                        continue
                    key = bench.digest([code_id, configs[metric], stamps, base["crop"], int(last), step, method])
                    cache = REPORT / "cache" / name / str(step) / method / metric / key
                    record = cache / "score.json"
                    row = dict(sequence=name, stride=step, m=step-1, selected_frames=len(ids),
                               last_scored_frame=int(last), duration_s=stop, method=method, metric=metric,
                               unit=bench.UNITS[metric], status="ok", value=None)
                    if record.exists():
                        saved = json.loads(record.read_text())
                        if not saved.get("array_path") or (REPORT / saved["array_path"]).is_file():
                            records.append(saved)
                            reused += 1
                            continue
                    try:
                        if not ground and not path.exists():
                            row["status"] = "missing_prediction"
                        elif method == "v2v":
                            row["status"] = "not_applicable"
                        else:
                            if events is None:
                                if ground:
                                    events = window.gt
                                else:
                                    meta = json.loads(metadata.read_text())
                                    if (Path(meta["source"]).parts[-3:] != ("eds", "test", name)
                                        or meta["indices"] != ids.tolist() or meta["crop"] != base["crop"]
                                        or meta["absolute_t0"] != ts[0] or meta["duration"] != ts[ids[-1]]-ts[0]
                                        or (step > 1 and not np.array_equal(meta["frame_times_s"], ts[ids]-ts[0]))):
                                        raise ValueError("Prediction metadata differs from selected frames or GT")
                                    stream = bench.read_events(path, meta)
                                    end = np.searchsorted(stream["t"], stop)
                                    events = {k: v[:end] for k,v in stream.items()}
                            nums, arrays, status = measure(metric, events, window, configs[metric]["options"])
                            row.update(nums, status=status)
                            if arrays:
                                with bench.atomic_file(cache / "arrays.npz", binary=True) as f:
                                    np.savez_compressed(f, **arrays)
                                row["array_path"] = str((cache / "arrays.npz").relative_to(REPORT))
                            computed += 1
                        if row["status"] != "missing_prediction":
                            with bench.atomic_file(record) as f:
                                json.dump(row, f, allow_nan=False)
                    except Exception as exc:
                        row.update(value=None, status="error", error=str(exc))
                        print(name, step, method, metric, exc, flush=True)
                    records.append(row)
        print(f"{name}: {reused} cached, {computed} computed", flush=True)
    return pd.DataFrame(records)

In [ ]:
if MODE not in {"auto", "plot_only"} or set(METRICS) - set(bench.UNITS) or not METRICS:
    raise ValueError("Check MODE and METRICS")
selection = bench.digest([sorted(SEQUENCES), sorted(METHODS), sorted(METRICS), EXPERIMENTS, DEVICE, WORKERS, VOXEL_BINS, str(DATA)])[:16]
directory = REPORT / selection
if MODE == "plot_only":
    scores = pd.read_csv(directory / "per_sequence.csv")
else:
    scores = run_scores()
    with bench.atomic_file(directory / "per_sequence.csv") as f:
        scores.to_csv(f, index=False)
valid = scores.value.where(scores.status.eq("ok"))
summary = (scores.assign(valid=valid).groupby(["metric","method","stride"], as_index=False)
           .agg(mean=("valid","mean"), n_valid=("valid","count"), n_total=("sequence","size")))
if MODE == "auto":
    with bench.atomic_file(directory / "summary.csv") as f:
        summary.to_csv(f, index=False)
print(directory)
display(summary)
display(scores)

## Interactive comparison
Choose the mean or one sequence from the menu. Hover over a point for coverage. Click legend entries to toggle methods. Recorded GT is shown for descriptive metrics. Summary means include only finite scores with status ok; inspect coverage when comparing methods.

In [ ]:
for metric in METRICS:
    fig = go.Figure()
    choices = ["Mean", *SEQUENCES]
    labels = list(summary.loc[summary.metric == metric, "method"].unique())
    for i, choice in enumerate(choices):
        for method in labels:
            if i == 0:
                group = summary[(summary.metric == metric) & (summary.method == method)].sort_values("stride")
                y = group["mean"]
                details = group.n_valid.astype(str) + "/" + group.n_total.astype(str) + " valid"
            else:
                group = scores[(scores.sequence == choice) & (scores.metric == metric) & (scores.method == method)].sort_values("stride")
                y = group.value.where(group.status == "ok")
                details = group.status
            gt = method == "ground_truth"
            fig.add_scatter(x=group.stride, y=y, customdata=details, name="Recorded GT" if gt else method,
                            mode="lines+markers", visible=i == 0, connectgaps=False,
                            line=dict(dash="dash" if gt else "solid", color="black" if gt else None),
                            hovertemplate="Stride %{x}<br>%{y:.5g}<br>%{customdata}<extra>%{fullData.name}</extra>")
    buttons = [dict(label=c, method="update", args=[dict(visible=[j//len(labels) == i for j in range(len(fig.data))])])
               for i,c in enumerate(choices)]
    title = "Median ISI per sequence" if metric == "isi_distribution" else metric
    fig.update_layout(title=title, xaxis=dict(title="Frame stride", tickvals=sorted(EXPERIMENTS)),
                      yaxis_title=bench.UNITS[metric], template="plotly_white",
                      updatemenus=[dict(buttons=buttons, x=0, y=1.2)])
    if metric == "event_count_ratio":
        fig.add_hline(y=1, line_dash="dot")
    fig.show()